# Hands-On Learning to Rank (LTR)


### Include required packages

In [15]:
import os
import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.backends.cudnn as cudnn
import wandb
import datetime

In [16]:
seed = 42
torch.cuda.manual_seed(seed)
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)
cudnn.deterministic = True
cudnn.benchmark = False

In [17]:
DEVICE = torch.device("cuda:0")
# DATA_DIR = '../data/MSLR-Web10K/Fold1/'
DATA_DIR = '../data/MQ2007/Fold1/'

MODE = 'pointwise'

FEAT_COUNT = 136 if 'MSLR' in DATA_DIR else 46
FEAT_SCALE = 1000
MB_SIZE = 1024
NUM_HIDDEN_NODES = 192 if 'MSLR' in DATA_DIR else 64
NUM_HIDDEN_LAYERS = 3
EPOCH_SIZE = 2048
NUM_EPOCHS = 50
LEARNING_RATE = 1e-4
DROPOUT_RATE = 0.0


DATA_FILE_TRAIN = os.path.join(DATA_DIR, 'train.txt')
DATA_FILE_TEST = os.path.join(DATA_DIR, 'test.txt')
MODEL_SAVE_PATH = f"ltr.{DATA_DIR.split('/')[-3]}.{NUM_EPOCHS}.pth"

### Configure the Weights and Biases

In [18]:
timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
wandb.init(project=f"ltr_{DATA_DIR.split('/')[-3]}", name=f"{MODE}_WeightedLoss_{NUM_EPOCHS}_{timestamp}")
wandb.config.update({
    'seed': seed,
    'feat_count': FEAT_COUNT,
    'feat_scale': FEAT_SCALE,
    'mb_size': MB_SIZE,
    'num_hidden_nodes': NUM_HIDDEN_NODES,
    'num_hidden_layers': NUM_HIDDEN_LAYERS,
    'epoch_size': EPOCH_SIZE,
    'num_epochs': NUM_EPOCHS,
    'learning_rate': LEARNING_RATE,
    'dropout_rate': DROPOUT_RATE,
    'data_file_train': DATA_FILE_TRAIN,
    'data_file_test': DATA_FILE_TEST,
    'model_save_path': MODEL_SAVE_PATH,
})
print(timestamp)

20240927-001124


### Define train and test data readers 

In [19]:
def parse_line(line):
    tokens = line.strip().split(' ')
    qid = -1
    feat = []
    label = int(tokens[0])
    
    for i in range(FEAT_COUNT):
        feat.append(0)
    
    for i in range(1, len(tokens)):
        sub_tokens = tokens[i].split(':')
        if sub_tokens[0] == 'qid':
            qid = int(sub_tokens[1])
        else:
            try:
                feat_idx = int(sub_tokens[0])
                feat_val = float(sub_tokens[1])
                feat[feat_idx - 1] = int(feat_val * FEAT_SCALE)
            except:
                pass
    return qid, label, feat

In [20]:
class DataReaderTrain():
    def __init__(self, data_file):
        self.data_file = data_file
        self.__load_data(self.data_file)

    def __iter__(self):
        self.__allocate_minibatch()
        return self

    def __load_data(self, data_file):
        self.data = {}
        with open(data_file, mode='r', encoding="utf-8") as f:
            for line in f:
                qid, label, feat = parse_line(line)
                if qid not in self.data:
                    self.data[qid] = {}
                if label not in self.data[qid]:
                    self.data[qid][label] = []
                self.data[qid][label].append(feat)
        
        self.data = {k: v for k, v in self.data.items() if len(v) > 1}
        self.qids = list(self.data.keys())
    
    def __allocate_minibatch(self):
        self.features = [np.zeros((MB_SIZE, FEAT_COUNT), dtype=np.float32) for i in range(2)]
        self.labels = np.zeros((MB_SIZE), dtype=np.int64)
        
    def __clear_minibatch(self):
        for i in range(2):
            self.features[i].fill(np.float32(0))
            
    def __next__(self):
        self.__clear_minibatch()
        qids = random.choices(self.qids, k=MB_SIZE)
        for i in range(MB_SIZE):
            labels = random.choices(list(self.data[qids[i]].keys()), k=2)
            labels.sort(reverse=True)
            for j in range(2):
                feats = self.data[qids[i]][labels[j]]
                feat = feats[random.randint(0, len(feats) - 1)]
                for k in range(FEAT_COUNT):
                    self.features[j][i, k] = feat[k]
        
        return [torch.from_numpy(self.features[i]).to(DEVICE) for i in range(2)], torch.from_numpy(self.labels).to(DEVICE)
    
    
class DataReaderTest():
    def __init__(self, data_file):
        self.data_file = data_file

    def __iter__(self):
        self.reader = open(self.data_file, mode='r', encoding="utf-8")
        self.__allocate_minibatch()
        return self
    
    def __allocate_minibatch(self):
        self.features = np.zeros((MB_SIZE, FEAT_COUNT), dtype=np.float32)
        self.labels = np.zeros((MB_SIZE), dtype=np.int64)
        
    def __clear_minibatch(self):
        self.features.fill(np.float32(0))
            
    def __next__(self):
        self.__clear_minibatch()
        qids = []
        labels = []
        cnt = 0
        for i in range(MB_SIZE):
            line = self.reader.readline()
            if line == '':
                raise StopIteration
                break
            
            qid, label, feat = parse_line(line)
            qids.append(qid)
            labels.append(label)
            
            for j in range(FEAT_COUNT):
                self.features[i, j] = feat[j]
            
            cnt += 1
        
        return torch.from_numpy(self.features).to(DEVICE), qids, labels, cnt

### Define the model

In [21]:
class MLP(nn.Module):

    def __init__(
            self,
            input_dim=137,
            index_dim=1,
            hidden_dim=2048,
            activation=nn.ReLU(),
    ):
        super().__init__()
        self.index_dim = index_dim
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.act = activation
        
        self.main = nn.Sequential(
            nn.Linear(input_dim + index_dim, hidden_dim),
            activation,
            nn.LayerNorm(hidden_dim),
            nn.Dropout(DROPOUT_RATE),
            nn.Linear(hidden_dim, hidden_dim),
            activation,
            nn.LayerNorm(hidden_dim),
            nn.Dropout(DROPOUT_RATE),
            nn.Linear(hidden_dim, hidden_dim),
            activation,
            nn.LayerNorm(hidden_dim),
            nn.Dropout(DROPOUT_RATE),
            nn.Linear(hidden_dim, input_dim),
        )
        

    def forward(self, features, t):
        input = features.view(-1, self.input_dim)
        t = t.view(-1, self.index_dim).float()

        # forward
        h = torch.cat([input, t], dim=1)  # concat
        output = self.main(h)  # forward
        return output.view(-1, self.input_dim)

In [22]:
class Diffusion(nn.Module):
    def __init__(self, model, num_features=137, n_times=1000, beta_minmax=[1e-4, 2e-2], device='cuda'):
        super(Diffusion, self).__init__()

        self.model = model.to(device)
        self.n_features = num_features
        self.n_times = n_times

        # define linear variance schedule(betas)
        beta_1, beta_T = beta_minmax
        betas = torch.linspace(start=beta_1, end=beta_T, steps=n_times).to(device)  # follows DDPM paper
        self.sqrt_betas = torch.sqrt(betas)

        # define alpha for forward diffusion kernel
        self.alphas = 1 - betas
        self.sqrt_alphas = torch.sqrt(self.alphas)
        alpha_bars = torch.cumprod(self.alphas, dim=0)
        self.sqrt_one_minus_alpha_bars = torch.sqrt(1 - alpha_bars)
        self.sqrt_alpha_bars = torch.sqrt(alpha_bars)

        self.device = device

    def extract(self, a, t, x_shape):
        """
            from lucidrains' implementation
                https://github.com/lucidrains/denoising-diffusion-pytorch/blob/beb2f2d8dd9b4f2bd5be4719f37082fe061ee450/denoising_diffusion_pytorch/denoising_diffusion_pytorch.py#L376
        """
        b, *_ = t.shape
        out = a.gather(-1, t)
        return out.reshape(b, *((1,) * (len(x_shape) - 1)))

    def scale_to_minus_one_to_one(self, x):
        # according to the DDPMs paper, normalization seems to be crucial to train reverse process network
        return x * 2 - 1

    def make_noisy(self, x_zeros, t):
        # perturb x_0 into x_t (i.e., take x_0 samples into forward diffusion kernels)
        epsilon = torch.randn_like(x_zeros).to(self.device)

        sqrt_alpha_bar = self.extract(self.sqrt_alpha_bars, t, x_zeros.shape)
        sqrt_one_minus_alpha_bar = self.extract(self.sqrt_one_minus_alpha_bars, t, x_zeros.shape)

        # Let's make noisy sample!: i.e., Forward process with fixed variance schedule
        #      i.e., sqrt(alpha_bar_t) * x_zero + sqrt(1-alpha_bar_t) * epsilon
        noisy_sample = x_zeros * sqrt_alpha_bar + epsilon * sqrt_one_minus_alpha_bar

        return noisy_sample.detach(), epsilon

    def forward(self, x_zeros, y):
        # x_zeros = self.scale_to_minus_one_to_one(x_zeros) # normalize to -1 to 1

        B, _ = x_zeros.shape

        # (1) randomly choose diffusion time-step
        t = torch.randint(low=0, high=self.n_times, size=(B,)).long().to(self.device)
        
        # t_value = torch.randint(low=0, high=self.n_times, size=(1,)).long().to(self.device)
        # t = t_value.repeat(B)

        # (2) forward diffusion process: perturb x_zeros with fixed variance schedule
        perturbed, epsilon = self.make_noisy(torch.cat((x_zeros, y), dim=1), t)

        # (3) predict epsilon(noise) given perturbed data at diffusion-timestep t.
        pred_epsilon = self.model(perturbed, t)

        return perturbed, epsilon, pred_epsilon

    def denoise_at_t(self, x_t, timestep, t):
        B, _ = x_t.shape
        if t > 1:
            z = torch.randn_like(x_t).to(self.device)
        else:
            z = torch.zeros_like(x_t).to(self.device)

        # at inference, we use predicted noise(epsilon) to restore perturbed data sample.
        epsilon_pred = self.model(x_t, timestep)

        alpha = self.extract(self.alphas, timestep, x_t.shape)
        sqrt_alpha = self.extract(self.sqrt_alphas, timestep, x_t.shape)
        sqrt_one_minus_alpha_bar = self.extract(self.sqrt_one_minus_alpha_bars, timestep, x_t.shape)
        sqrt_beta = self.extract(self.sqrt_betas, timestep, x_t.shape)

        # denoise at time t, utilizing predicted noise
        x_t_minus_1 = 1 / sqrt_alpha * (x_t - (1 - alpha) / sqrt_one_minus_alpha_bar * epsilon_pred) + sqrt_beta * z

        return x_t_minus_1.clamp(-1., 1)

    def predict(self, x):
        # start from random noise vector, x_0 (for simplicity, x_T declared as x_t instead of x_T)
        x_t = torch.randn((x.size(0), self.n_features)).to(self.device)

        # autoregressively denoise from x_T to x_0

        # Denoise
        for t in range(self.n_times - 1, -1, -1):
            for j in range(5): ## Harmonization
                timestep = torch.tensor([t]).repeat_interleave(x.size(0), dim=0).long().to(self.device)
                x_t_1_unknown = self.denoise_at_t(x_t, timestep, t)
                if t > 0:
                    x_t_1_known, epsilon = self.make_noisy(x, timestep - 1)
                else:
                    x_t_1_known = x
                    epsilon = torch.zeros_like(x_t_1_known[:, :self.n_features]).to(self.device)
                x_t_1 = torch.cat((x_t_1_known[:, :(self.n_features - 1)], x_t_1_unknown[:, (self.n_features - 1):]), dim=1)
                if j < 4 and t > 0:
                    # Add noise for one step
                    x_t = self.sqrt_alphas[t] * x_t_1 + self.sqrt_betas[t] * epsilon.to(self.device)
                else:
                    x_t = x_t_1

        x_0 = x_t
        return x_0

In [23]:
class WeightedMSELoss(nn.Module):
    def __init__(self, weight):
        super(WeightedMSELoss, self).__init__()
        self.weight = weight

    def forward(self, input, target):
        # Create a weight tensor with the same shape as input/target
        # Set all weights to 1, except for the last feature which is multiplied by the specified weight
        weights = torch.ones_like(input)
        weights[:, -1] = self.weight

        # Compute the squared difference
        squared_diff = (input - target) ** 2

        # Apply the weights and take the mean
        weighted_loss = weights * squared_diff
        return weighted_loss.mean()

### Train and evaluate

In [24]:
READER_TRAIN = DataReaderTest(DATA_FILE_TRAIN)
READER_TRAIN_ITER = iter(READER_TRAIN)
READER_TEST = DataReaderTest(DATA_FILE_TEST)

model = MLP(input_dim=FEAT_COUNT + 1, hidden_dim=NUM_HIDDEN_NODES)
net = Diffusion(model, num_features=FEAT_COUNT + 1, device=DEVICE)
optimizer = optim.Adam(net.parameters(), lr=LEARNING_RATE)
if MODE == 'pairwise':
    criterion = nn.MarginRankingLoss(margin=1.0)
elif MODE == 'pointwise':
    # criterion = nn.MSELoss()
    criterion = WeightedMSELoss(weight=40.0)

In [25]:
iterator_len = 0
for features, _, labels, _ in READER_TRAIN_ITER:
    iterator_len += 1
    
print(f"Data loaded. Train data batch count: {iterator_len}")

Data loaded. Train data batch count: 41


In [26]:
def train(net):
    train_loss = 0.0
    net.train()
    e_size = 0
    
    for _ in range(max(1, round(EPOCH_SIZE / iterator_len))):
        for features, _, labels, _ in READER_TRAIN_ITER:
            e_size += 1
            #Read in a new mini-batch of data!
            # features, labels = next(READER_TRAIN_ITER) #Read in a new mini-batch of data!
            labels = torch.tensor(labels).to(DEVICE)
            labels = labels.unsqueeze(1)
            
            if MODE == 'pairwise':
                noisy_input0, epsilon0, pred_epsilon0 = net(features[0], labels)
                noisy_input1, epsilon1, pred_epsilon1 = net(features[1], labels)
                loss = ...
            elif MODE == 'pointwise':
                noisy_input, epsilon, pred_epsilon = net(features, labels)
                # noisy_input, epsilon, pred_epsilon = net(features[0], labels)
                loss = criterion(pred_epsilon, epsilon)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
    
    return train_loss / e_size

# def train(net):
#     train_loss = 0.0
#     net.train()
#     e_size = 0
#     count = 0
#     for _ in range(EPOCH_SIZE):
#     # for features, _, labels, _ in READER_TRAIN_ITER:
#         e_size += 1
#         #Read in a new mini-batch of data!
#         features, labels = next(READER_TRAIN_ITER) #Read in a new mini-batch of data!
#         # labels = torch.tensor(labels).to(DEVICE)
#         labels = labels.unsqueeze(1)
        
#         if MODE == 'pairwise':
#             noisy_input0, epsilon0, pred_epsilon0 = net(features[0], labels)
#             noisy_input1, epsilon1, pred_epsilon1 = net(features[1], labels)
#             loss = ...
#         elif MODE == 'pointwise':
#             noisy_input, epsilon, pred_epsilon = net(features[0], labels)
#             count += 1
#             # noisy_input, epsilon, pred_epsilon = net(features[0], labels)
#             loss = criterion(pred_epsilon, epsilon)
        
#         optimizer.zero_grad()
#         loss.backward()
#         optimizer.step()
#         train_loss += loss.item()
#     print(count)
#     return train_loss / e_size


def test(net, epoch, train_loss):
    net.eval()
    reader_test_iter = iter(READER_TEST)
    c = False
    
    with torch.no_grad():
        results = {}
        for features, qids, labels, cnt in reader_test_iter:
            random_labels = torch.randint(0, 4 if 'MSLR' in DATA_DIR else 2, (cnt, 1)).to(DEVICE)
            out = net.predict(torch.cat((features, random_labels), dim=1))[:, -1]
            out = out.unsqueeze(1).cpu()
            if c:
                print(out)
                print(out.shape)
                c = False
            row_cnt = len(qids)
            for i in range(row_cnt):
                if qids[i] not in results:
                    results[qids[i]] = []
                results[qids[i]].append((labels[i], out[i][0]))
        
        avgp = 0
        avgndcg = 0
        for qid, docs in results.items():
            dcg = 0
            ranked = sorted(docs, key=lambda x: x[1], reverse=True)

            relevant_in_top10 = sum(1 for doc in ranked[:10] if doc[0] > 0)
            p = relevant_in_top10 / min(10, len(ranked))
            avgp += p
            
            for i in range(min(10, len(ranked))):
                rank = i + 1
                label = ranked[i][0]
                dcg += ((2**label - 1) / math.log2(rank + 1))
            idcg = 0
            ranked = sorted(docs, key=lambda x: x[0], reverse=True)
            for i in range(min(10, len(ranked))):
                rank = i + 1
                label = ranked[i][0]
                idcg += ((2**label - 1) / math.log2(rank + 1))
            if idcg > 0:
                avgndcg += (dcg / idcg)
        avgp /= len(results)
        avgndcg /= len(results)
        print(f'epoch:{epoch}, loss: {train_loss}, avgp: {avgp}, avgndcg: {avgndcg}')
        
    return avgp, avgndcg

In [27]:
print('Dataset: {}'.format(DATA_DIR))
print(f"Model has {(sum(p.numel() for p in model.parameters() if p.requires_grad)):,} trainable parameters")
print('Learning rate: {}'.format(LEARNING_RATE))
print('Mode: {} - Generative'.format(MODE))

test(net, 0, 'n/a')
for epoch in range(NUM_EPOCHS):
    train_loss = train(net)
    avgp, avgndcg = test(net, epoch + 1, str(train_loss))
    
    wandb.log({
        'epoch': epoch,
        'train_loss': train_loss,
        'avgp': avgp,
        'avgndcg': avgndcg,
    })
    
wandb.finish()
# save model
torch.save(net.state_dict(), MODEL_SAVE_PATH)

print('Finished training')

Dataset: ../data/MQ2007/Fold1/
Model has 14,895 trainable parameters
Learning rate: 0.0001
Mode: pointwise - Generative
epoch:0, loss: n/a, avgp: 0.26859756097560966, avgndcg: 0.25749393028852047
epoch:1, loss: 1.8022334973405048, avgp: 0.2670731707317074, avgndcg: 0.25812643096923643
epoch:2, loss: 1.3895883931183233, avgp: 0.28018292682926815, avgndcg: 0.28082886860871004
epoch:3, loss: 1.228923048682329, avgp: 0.2777439024390245, avgndcg: 0.2681583403753889
epoch:4, loss: 1.1752908801450963, avgp: 0.29603658536585364, avgndcg: 0.28953343568685797
epoch:5, loss: 1.1475399658156604, avgp: 0.3033536585365851, avgndcg: 0.3046123575192877
epoch:6, loss: 1.1247741818428039, avgp: 0.2865853658536582, avgndcg: 0.2796930258097691
epoch:7, loss: 1.1059944943102396, avgp: 0.29634146341463397, avgndcg: 0.30942648111042903
epoch:8, loss: 1.0903126760226924, avgp: 0.29878048780487787, avgndcg: 0.2982879933285902
epoch:9, loss: 1.078521647511459, avgp: 0.2966463414634147, avgndcg: 0.30175987175972

In [14]:
wandb.finish()

avgndcg,▂▂▃▁▃▃▄▄▃▂▄▄▃▄▃▆▅▅▇▇▆▆▆▆▆▇▇▇▅▆▆█▇▇▅
avgp,▃▂▃▁▃▁▃▄▃▂▂▄▃▄▄▇▆▇▆▇▆▆▇▆▆▇▇▇▆▆▆█▆▇▆
epoch,▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇███
train_loss,███▆▅▄▄▄▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
avgndcg,0.29292
avgp,0.29848
epoch,34
train_loss,1.06816


In [43]:
import torch.nn.functional as F

def test_overfit_to_train_noisy(net, reader_train):
    net.eval()
    total_mse = 0
    total_samples = 0
    
    with torch.no_grad():
        for features, _, labels, cnt in reader_train:
            labels = torch.tensor(labels).to(DEVICE).unsqueeze(1)
            input_data = torch.cat((features, labels), dim=1)
            
            # Sample random timesteps
            t = torch.randint(0, net.n_times, (cnt,), device=DEVICE)
            
            # Add noise
            noisy_input, epsilon = net.make_noisy(input_data, t)
            
            # Predict epsilon
            pred_epsilon = net.model(noisy_input, t)
            
            # Calculate MSE
            mse = F.mse_loss(pred_epsilon, epsilon, reduction='sum')
            total_mse += mse.item()
            total_samples += cnt
    
    avg_mse = total_mse / total_samples
    return avg_mse

def test_overfit_to_train_full_noise(net, reader_train):
    net.eval()
    total_mse = 0
    total_samples = 0
    
    with torch.no_grad():
        for features, _, labels, cnt in reader_train:
            labels = torch.tensor(labels).to(DEVICE).unsqueeze(1)
            input_data = torch.cat((features, labels), dim=1)
            
            # Add full noise (T_max)
            t_max = torch.full((cnt,), net.n_times - 1, device=DEVICE)
            noisy_input, _ = net.make_noisy(input_data, t_max)
            
            # Recover original vector
            recovered = net.predict(noisy_input)
            
            # Calculate MSE
            mse = F.mse_loss(recovered, input_data, reduction='sum')
            total_mse += mse.item()
            total_samples += cnt
    
    avg_mse = total_mse / total_samples
    return avg_mse


print("Running overfitting tests on training data...")
# Test 1: Overfit to train (noisy)
reader_train_test = DataReaderTest(DATA_FILE_TRAIN)
noisy_mse = test_overfit_to_train_noisy(net, reader_train_test)
print(f"Test 1 - MSE for noisy prediction: {noisy_mse:.6f}")

# Test 2: Overfit to train (full noise)
# full_noise_mse = test_overfit_to_train_full_noise(net, reader_train_test)
# print(f"Test 2 - MSE for full noise recovery: {full_noise_mse:.6f}")

Running overfitting tests on training data...
Test 1 - MSE for noisy prediction: 36.893138
